# N13 · RoPE 与 YaRN 数学

**关联 lab**: L04.5

**学习目标**: 把 RoPE 的旋转编码、频率衰减曲线、YaRN 缩放公式从 paper 公式变成可断言的 numpy 复现。理解 "训练 4K + YaRN scale=4 推理 16K" 为什么能 work、什么时候不能。

**No-GPU 可完成度**: 100%。

**对应 MiniInfra**:
- `mini_infra/megatron/core/context_parallel/rope.py`
- `mini_infra/megatron/core/context_parallel/yarn.py`

**对应真实源码**: `github_repo/Megatron-LM/megatron/core/models/common/embeddings/rotary_pos_embedding.py`

## 1. RoPE 频率公式

RoPE 的核心是给位置 m 的 query/key 旋转一个角度 m·θᵢ，其中：

$$\theta_i = \text{base}^{-2i/d}, \quad i = 0, 1, \dots, d/2 - 1$$

- 低 i（小 i） → 高频，相位变化快
- 高 i（接近 d/2） → 低频，相位变化慢
- base=10000 是 LLaMA/GPT 默认；base=1000000 是 NTK-aware 的常见选择

用 `inverse_frequencies` 看具体数字：

In [ ]:
import sys, math
from pathlib import Path
ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from mini_infra.megatron.core.context_parallel.rope import (
    inverse_frequencies, rope_angles, apply_rope_pairs, rope_summary
)

for base in [10000, 1000000]:
    freqs = inverse_frequencies(dim=128, base=base)
    print(f'base={base:>8}  high-freq θ_0 = {freqs[0]:.6f}  low-freq θ_{len(freqs)-1} = {freqs[-1]:.2e}')
    print(f'           频率范围跨 {freqs[0]/freqs[-1]:.2e} 个数量级')

## 2. 旋转操作：复数视角

把 query 的相邻一对 (x_even, x_odd) 看成复数 z = x_even + i·x_odd，旋转角度 m·θ：

$$z' = z \cdot e^{i m \theta} = (x_{even} \cos(m\theta) - x_{odd} \sin(m\theta)) + i(x_{even} \sin(m\theta) + x_{odd} \cos(m\theta))$$

用 `rotate_pair` 与 `apply_rope_pairs` 验证：

In [ ]:
values = [1.0, 0.0, 0.5, 0.5, -1.0, 1.0, 0.3, -0.7]
for position in [0, 100, 1000, 4096]:
    rotated = apply_rope_pairs(values, position=position)
    print(f'pos={position:>5}  {[round(v, 4) for v in rotated]}')

**关键不变量**: 旋转后的 Q·K 内积只依赖位置差 m-n，不依赖绝对位置。这是 RoPE 比绝对位置编码 robust 的原因。

Sanity check: position=0 时旋转角度 = 0，应该等于原值：

In [ ]:
rotated_at_zero = apply_rope_pairs(values, position=0)
max_diff = max(abs(a - b) for a, b in zip(values, rotated_at_zero))
print('position=0 旋转后与原值的 max_abs_err =', max_diff, '(应当 == 0)')

## 3. 训练 ctx 的最大相位

训练时 ctx_train=4096，最高频位 i=0 在 position=4095 时的相位：

$$4095 \times \theta_0 = 4095 \times 1.0 = 4095 \text{ rad}$$

这意味着模型见过的最大旋转角度。推理超过 ctx_train 时，position=8192 的相位会到 8192 rad，而模型没见过这个区间——这就是外推失败的根因。

In [ ]:
for ctx in [4096, 8192, 16384, 32768]:
    summary = rope_summary(seq_len=ctx, dim=128, base=10000)
    print(f'ctx={ctx:>5}  max_angle={summary["max_angle"]:>10.1f} rad  '
          f'高频 θ_0={summary["rope_freq_max"]:.4f}  低频={summary["rope_freq_min"]:.2e}')

**直觉**: 训练 ctx=4096 时模型见过 max_angle ≈ 4095 rad。推理 32768 时跳到 32767 rad，差 8 倍。模型没法直接外推。

**两种主流外推方法**：
1. **PI (Position Interpolation)**: 把推理位置压缩回训练范围，`pos' = pos / scale`。简单但效果一般。
2. **YaRN (Yet another RoPE eN extension)**: 对低频/高频位分段缩放 + 调整 attention temperature。效果最好但需要少量 finetune。

## 4. YaRN 缩放公式

`mini_infra/.../yarn.py` 的简化版：

$$\text{scale} = 1 + \frac{\log(\text{ctx}_{\text{target}} / \text{ctx}_{\text{train}})}{\log(\beta_{\text{fast}})}$$

$$\text{attention\_temperature} = 1 + 0.1 (\text{scale} - 1)$$

其中 β_fast 默认 32（YaRN 论文）。

scale 用于在 RoPE 频率中缩放低频位：θ_i' = θ_i / scale（高频位几乎不变，因为 NTK-aware）。

In [ ]:
from mini_infra.megatron.core.context_parallel.yarn import yarn_scale, yarn_temperature, yarn_summary

for target in [4096, 8192, 16384, 32768, 131072]:
    s = yarn_scale(train_seq_len=4096, target_seq_len=target)
    t = yarn_temperature(s)
    print(f'train=4096 → target={target:>6}  yarn_scale={s:.4f}  attention_temp={t:.4f}')

**关键观察**: scale 不是线性增长。从 4K→8K（2x）到 4K→32K（8x），scale 只从 1.20 到 1.60。

这说明 YaRN 是 "温和" 的外推：不让推理位置远离训练分布。

## 5. PI vs YaRN 对照

用同一 ckpt（训练 ctx=4K）模拟在 16K 上推理时的位置编码差异：

- PI: `pos' = pos × ctx_train / ctx_target = pos / 4`
- YaRN: `pos' = pos / scale`，scale ≈ 1.4

PI 把所有位置都压回 4K 内；YaRN 只温和压缩，让模型仍能区分 "近" 与 "远"。

In [ ]:
ctx_train, ctx_target = 4096, 16384
pi_scale = ctx_target / ctx_train  # = 4
yarn_s = yarn_scale(ctx_train, ctx_target)

print(f'PI scale     = {pi_scale}')
print(f'YaRN scale   = {yarn_s:.4f}')
print()
print(f'{"raw_pos":>10} {"PI":>10} {"YaRN":>10}')
for raw_pos in [0, 1000, 4000, 8000, 16000]:
    pi_pos = raw_pos / pi_scale
    yarn_pos = raw_pos / yarn_s
    print(f'{raw_pos:>10} {pi_pos:>10.1f} {yarn_pos:>10.1f}')

**结论**: PI 在 16K 上把 raw_pos=16000 压到 4000（保留在 ctx_train 内）；YaRN 只压到 11400+（仍在外推）但搭配 attention_temperature 修正可保留长程能力。

PI 简单但 "分辨率" 损失大（远距离细节模糊）；YaRN 复杂但保留更多信息。

## 6. 自检问题（运行后回答）

1. base=10000 与 base=1000000 在 dim=128 时高频/低频跨度有何差异？哪个更适合长 ctx？
2. 证明 RoPE 旋转后 Q·K 内积只依赖 m-n。（提示：(z₁ e^{im}) · (z₂ e^{-in}) = z₁z₂ e^{i(m-n)}）
3. 训练 ctx=4K，推理 32K 时 YaRN scale 大约是多少？attention_temperature 多少？
4. PI 与 YaRN 在 16K 推理时把 raw_pos=16000 各压到多少？哪个更激进？
5. 如果训练 ckpt 用了 base=1000000（NTK-aware），推理时还应该开 YaRN scale 吗？为什么？

## 7. 与 lab 对接

下一步 lab 任务：在 Megatron 上跑 seq=4K/16K/64K，每组开/不开 YaRN，对照 eval ppl。

Smoke 命令：
```bash
python labs/l09_long_context_cp/scripts/eval_yarn.py --scale 1.4 --eval-ctx 16384
```

**关键 ticket**: 看到推理输出胡乱时一定先看 `tickets/yarn_scale_misuse_002.yaml`——99% 是训练/推理 RoPE 配置不一致。